# WiFi Sonar Training Pipeline
GPU-ready notebook for presence → activity → pose models (Hernandez Thesis Ch. 6, COMST 2023 Sec. 4).

## 0. Environment
- Requires PyTorch 2.2+, CUDA 11+, sklearn, pandas.
- Run inside the `RV2` repo root so relative data paths resolve.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print(torch.__version__)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU-only')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Data Loader (PCA features only)

In [ ]:
data_dir = Path('data/raw')
session_files = sorted(data_dir.glob('*.csv'))
assert session_files, 'No CSV files found in data/raw. Run collector first.'

PC_COLUMNS = [f'pc{i}' for i in range(1, 11)]
META_COLUMNS = ['host_ts', 'label', 'phase', 'presence_flag']

frames = [pd.read_csv(path, usecols=PC_COLUMNS + META_COLUMNS) for path in session_files]
raw_df = pd.concat(frames, ignore_index=True)
print(f'Loaded {len(raw_df)} samples from {len(session_files)} sessions')

## 2. Presence Detector (binary)\nUse PCA features → 2-layer MLP (Thesis p. 176).

In [ ]:
X = raw_df[PC_COLUMNS].values.astype(np.float32)
y = raw_df['presence_flag'].values.astype(np.float32)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42)

class PresenceNet(torch.nn.Module):
    def __init__(self, input_dim=10, hidden_dim=64):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.1),
            torch.nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x)

model = PresenceNet().to(device)
criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

train_tensor = torch.tensor(X_train).to(device)
train_labels = torch.tensor(y_train).unsqueeze(1).to(device)
val_tensor = torch.tensor(X_val).to(device)
val_labels = torch.tensor(y_val).unsqueeze(1).to(device)

for epoch in range(30):
    model.train()
    optimizer.zero_grad()
    logits = model(train_tensor)
    loss = criterion(logits, train_labels)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_logits = model(val_tensor)
        val_loss = criterion(val_logits, val_labels)
        val_pred = (torch.sigmoid(val_logits) > 0.5).float()
        val_acc = (val_pred == val_labels).float().mean().item()
    if epoch % 5 == 0 or epoch == 29:
        print(f'Epoch {epoch:02d} | train {loss.item():.4f} | val {val_loss.item():.4f} | acc {val_acc:.3f}')

## 3. Activity Classifier (walk vs fall vs welding vs grinding)

In [ ]:
activity_mask = raw_df['phase'].eq('run')
activity_df = raw_df[activity_mask].copy()
activity_df['activity'] = activity_df['label'].str.lower()
activity_df = activity_df[activity_df['activity'].isin(['walk', 'fall', 'welding', 'grinding'])]

enc = LabelEncoder()
y_act = enc.fit_transform(activity_df['activity'])
X_act = activity_df[PC_COLUMNS].values.astype(np.float32)

Xa_train, Xa_val, ya_train, ya_val = train_test_split(X_act, y_act, test_size=0.2, random_state=0)

class ActivityNet(torch.nn.Module):
    def __init__(self, input_dim=10, classes=4):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, 96),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.2),
            torch.nn.Linear(96, 48),
            torch.nn.ReLU(),
            torch.nn.Linear(48, classes),
        )

    def forward(self, x):
        return self.net(x)

act_model = ActivityNet(classes=len(enc.classes_)).to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(act_model.parameters(), lr=5e-4)

Xa_train_t = torch.tensor(Xa_train).to(device)
ya_train_t = torch.tensor(ya_train).to(device)
Xa_val_t = torch.tensor(Xa_val).to(device)
ya_val_t = torch.tensor(ya_val).to(device)

for epoch in range(50):
    act_model.train()
    optimizer.zero_grad()
    logits = act_model(Xa_train_t)
    loss = criterion(logits, ya_train_t)
    loss.backward()
    optimizer.step()

    act_model.eval()
    with torch.no_grad():
        val_logits = act_model(Xa_val_t)
        val_loss = criterion(val_logits, ya_val_t)
        preds = val_logits.argmax(dim=1)
        val_acc = (preds == ya_val_t).float().mean().item()
    if epoch % 10 == 0 or epoch == 49:
        print(f'Epoch {epoch:02d} | train {loss.item():.4f} | val {val_loss.item():.4f} | acc {val_acc:.3f}')

torch.save({
    'presence_net': model.state_dict(),
    'activity_net': act_model.state_dict(),
    'label_encoder': enc.classes_.tolist(),
}, Path('artifacts') / 'wifi_sonar_ckpt.pt')

## 4. Pose Prototype (coarse stance regression)

In [ ]:
pose_labels = {'walk': 0, 'fall': 2, 'breathing': 2, 'welding': 0, 'grinding': 0}
pose_df = raw_df.copy()
pose_df['pose'] = pose_df['label'].map(pose_labels)
pose_df = pose_df.dropna(subset=['pose'])

if len(pose_df) > 100:
    X_pose = pose_df[PC_COLUMNS].values.astype(np.float32)
    y_pose = pose_df['pose'].astype(int).values
    Xp_train, Xp_val, yp_train, yp_val = train_test_split(X_pose, y_pose, test_size=0.2, random_state=1)

    pose_model = ActivityNet(classes=3).to(device)
    opt = torch.optim.AdamW(pose_model.parameters(), lr=5e-4)
    crit = torch.nn.CrossEntropyLoss()

    Xp_train_t = torch.tensor(Xp_train).to(device)
    yp_train_t = torch.tensor(yp_train).to(device)
    Xp_val_t = torch.tensor(Xp_val).to(device)
    yp_val_t = torch.tensor(yp_val).to(device)

    for epoch in range(30):
        pose_model.train()
        opt.zero_grad()
        logits = pose_model(Xp_train_t)
        loss = crit(logits, yp_train_t)
        loss.backward()
        opt.step()
        if epoch % 10 == 0:
            pose_model.eval()
            with torch.no_grad():
                v_logits = pose_model(Xp_val_t)
                v_acc = (v_logits.argmax(1) == yp_val_t).float().mean().item()
            print(f'Pose epoch {epoch} acc={v_acc:.3f}')
else:
    print('Pose dataset too small; collect more labeled samples.')